# Uji Wilcoxon Signed Rank

## Load Library

In [ ]:
from tabulate import tabulate
from scipy.stats import wilcoxon, norm
import pandas as pd

## Define Critical Value from Wilcoxon Table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def get_critical_value(n, alpha):
    file_path = "/content/drive/MyDrive/Wilcoxon-Signed-Ranks-Table.xlsx"
    sheet_name = "Signed Rank Table"
    df = pd.read_excel(file_path, sheet_name=sheet_name, usecols="A:H", skiprows=4, nrows=47)
    df.columns = ["n", "alpha_0.2", "alpha_0.1", "alpha_0.05", "alpha_0.02", "alpha_0.01", "alpha_0.002", "alpha_0.001"]
    alpha_col = f"alpha_{alpha}" if f"alpha_{alpha}" in df.columns else None

    if alpha_col is None or n not in df["n"].values:
        return "Gunakan distribusi normal atau tabel Wilcoxon lainnya"

    return df.loc[df["n"] == n, alpha_col].values[0]

## Define Function

### Two-Tail

In [ ]:
def wilcoxon_test(test_type, data1, data2=None, median_assumption=None, alpha=None):

    if test_type == "one_sample":
        print("1. Hipotesis:")
        print(f"H0: Median sampel = {median_assumption}")
        print(f"H1: Median sampel ≠ {median_assumption}")

        differences = [x - median_assumption for x in data1]
        abs_differences = [abs(diff) for diff in differences]

        zero_data = [(x, di, abs_di) for x, di, abs_di in zip(data1, differences, abs_differences) if di == 0]
        nonzero_data = [(x, di, abs_di) for x, di, abs_di in zip(data1, differences, abs_differences) if di != 0]

        sorted_data = sorted(nonzero_data, key=lambda x: x[2])
        ranks = {}

        i = 1
        while i <= len(sorted_data):
            same_values = [sorted_data[i - 1][2]]
            j = i
            while j < len(sorted_data) and sorted_data[j][2] == sorted_data[i - 1][2]:
                same_values.append(sorted_data[j][2])
                j += 1
            avg_rank = sum(range(i, j + 1)) / len(same_values)
            for k in range(i, j + 1):
                ranks[k] = avg_rank
            i = j + 1

        table_data = []
        T_positive = 0
        T_negative = 0
        for i, (x, di, abs_di) in enumerate(sorted_data, start=1):
            rank = ranks[i]
            sign = '+' if di > 0 else '-'
            table_data.append([x, di, abs_di, rank, sign])
            if di > 0:
                T_positive += rank
            else:
                T_negative += rank

        for x, di, abs_di in zero_data:
            table_data.append([x, di, abs_di, '-', '-'])

        headers = ["Data", "di", "|di|", "Rank", "Signed Rank"]
        print(tabulate(table_data, headers=headers, tablefmt="grid"))

        T = min(T_positive, T_negative)
        n = len(sorted_data)

        if n <= 25:
            print("2. Kriteria Uji:")
            print("Jika T < nilai kritis d, maka tolak H0")

            critical_value = get_critical_value(n, alpha)

            print("\n3. Statistika Uji:")
            print(f"Jumlah peringkat positif: {T_positive}")
            print(f"Jumlah peringkat negatif: { T_negative}")
            print(f"Statistik uji Wilcoxon (T): {T}")
            print(f"Alpha: {alpha}")
            print(f"Nilai kritis Wilcoxon (d): {critical_value}")

            print("\n4. Penarikan Kesimpulan")
            if isinstance(critical_value, (int, float)) and T < critical_value:
                print(f"Karena T: {T} < d: {critical_value}")
                print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan.")
            else:
                print(f"Karena T: {T} >= d: {critical_value}")
                print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan.")

        else:
            print("2. Kriteria Uji:")
            print("Jika |Z hitung| > Z tabel, maka tolak H0")

            mu_T = n * (n + 1) / 4
            sigma_T = ((n * (n + 1) * (2 * n + 1)) / 24) ** 0.5
            z = (T - mu_T) / sigma_T
            z_critical = norm.ppf(1 - alpha / 2)

            print("\n3. Statistika Uji:")
            print(f"Jumlah peringkat positif: {T_positive}")
            print(f"Jumlah peringkat negatif: {T_negative}")
            print(f"Statistik uji Wilcoxon (T): {T}")
            print(f"Alpha: {alpha}")

            print(f"Z_hitung: {z}, Z_tabel: ±{z_critical}")

            print("\n4. Penarikan Kesimpulan")
            if abs(z) > z_critical:
                print(f"Karena absolute Z-hitung: {z} > Z-tabel {z_critical}")
                print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan.")
            else:
                print(f"Karena absolute Z-hitung: {z} <= Z-tabel {z_critical}")
                print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan.")

    elif test_type == "paired":
        if len(data1) != len(data2):
            raise ValueError("Panjang data1 dan data2 harus sama")

        print("1. Hipotesis:")
        print("   H0: Tidak ada perbedaan yang signifikan antara kedua sampel")
        print("   H1: Ada perbedaan yang signifikan antara kedua sampel")

        differences = [x - y for x, y in zip(data1, data2)]
        abs_differences = [abs(diff) for diff in differences]

        zero_data = [(d1, d2, di, abs_di) for d1, d2, di, abs_di in zip(data1, data2, differences, abs_differences) if di == 0]
        nonzero_data = [(d1, d2, di, abs_di) for d1, d2, di, abs_di in zip(data1, data2, differences, abs_differences) if di != 0]

        sorted_data = sorted(nonzero_data, key=lambda x: x[3])
        ranks = {}

        i = 1
        while i <= len(sorted_data):
            same_values = [sorted_data[i - 1][3]]
            j = i
            while j < len(sorted_data) and sorted_data[j][3] == sorted_data[i - 1][3]:
                same_values.append(sorted_data[j][3])
                j += 1
            avg_rank = sum(range(i, j + 1)) / len(same_values)
            for k in range(i, j + 1):
                ranks[k] = avg_rank
            i = j + 1

        table_data = []
        T_positive = 0
        T_negative = 0
        for i, (d1, d2, di, abs_di) in enumerate(sorted_data, start=1):
            rank = ranks[i]
            sign = '+' if di > 0 else '-'
            table_data.append([d1, d2, di, abs_di, rank, sign])
            if di > 0:
                T_positive += rank
            else:
                T_negative += rank

        for d1, d2, di, abs_di in zero_data:
            table_data.append([d1, d2, di, abs_di, '-', '-'])

        headers = ["Data1", "Data2", "di", "|di|", "Rank", "Signed Rank"]
        print(tabulate(table_data, headers=headers, tablefmt="grid"))

        T = min(T_positive, T_negative)
        n = len(nonzero_data)

        if n<= 25:
            print("2. Kriteria Uji:")
            print("Jika T < nilai kritis d, maka tolak H0")

            critical_value = get_critical_value(n, alpha)

            print("\n3. Statistika Uji:")
            print(f"Jumlah peringkat positif: {T_positive}")
            print(f"Jumlah peringkat negatif: {T_negative}")
            print(f"Statistik uji Wilcoxon (T): {T}")
            print(f"Alpha: {alpha}")
            print(f"Nilai kritis Wilcoxon (d): {critical_value}")

            print("\n4. Penarikan Kesimpulan")
            if isinstance(critical_value, (int, float)) and T < critical_value:
                print(f"Karena T: {T} < d: {critical_value}")
                print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan.")
            else:
                print(f"Karena T: {T} >= d: {critical_value}")
                print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan.")

        else:
            print("2. Kriteria Uji:")
            print("Jika |Z hitung| > Z tabel, maka tolak H0")

            mu_T = n * (n + 1) / 4
            sigma_T = ((n * (n + 1) * (2 * n + 1)) / 24) ** 0.5
            z = (T - mu_T) / sigma_T
            z_critical = norm.ppf(1 - alpha / 2)

            print("\n3. Statistika Uji:")
            print(f"Jumlah peringkat positif: {T_positive}")
            print(f"Jumlah peringkat negatif: {T_negative}")
            print(f"Statistik uji Wilcoxon (T): {T}")
            print(f"Alpha: {alpha}")

            print(f"Z_hitung: {z}, Z_tabel: ±{z_critical}")

            print("\n4. Penarikan Kesimpulan")
            if abs(z) > z_critical:
                print(f"Karena absolute Z-hitung: {z} > Z-tabel {z_critical}")
                print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan.")
            else:
                print(f"Karena absolute Z-hitung: {z} <= Z-tabel {z_critical}")
                print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan.")


    else:
        print("Jenis uji tidak valid. Gunakan 'one_sample' atau 'paired'.")

# two tail n one tail

In [ ]:
import math
from collections import Counter
import numpy as np

In [ ]:
def wilcoxon_test(test_type, data1, data2=None, median_assumption=None, alpha=None, alternative=None):

    if test_type == "one_sample":

        differences = [x - median_assumption for x in data1]
        abs_differences = [abs(diff) for diff in differences]

        zero_data = [(x, di, abs_di) for x, di, abs_di in zip(data1, differences, abs_differences) if di == 0]
        nonzero_data = [(x, di, abs_di) for x, di, abs_di in zip(data1, differences, abs_differences) if di != 0]

        sorted_data = sorted(nonzero_data, key=lambda x: x[2])
        ranks = {}

        i = 1
        while i <= len(sorted_data):
            same_values = [sorted_data[i - 1][2]]
            j = i
            while j < len(sorted_data) and sorted_data[j][2] == sorted_data[i - 1][2]:
                same_values.append(sorted_data[j][2])
                j += 1
            avg_rank = sum(range(i, j + 1)) / len(same_values)
            for k in range(i, j + 1):
                ranks[k] = avg_rank
            i = j + 1

        table_data = []
        T_positive = 0
        T_negative = 0
        for i, (x, di, abs_di) in enumerate(sorted_data, start=1):
            rank = ranks[i]
            sign = '+' if di > 0 else '-'
            table_data.append([x, di, abs_di, rank, sign])
            if di > 0:
                T_positive += rank
            else:
                T_negative += rank

        for x, di, abs_di in zero_data:
            table_data.append([x, di, abs_di, '-', '-'])

        headers = ["Data", "di", "|di|", "Rank", "Signed Rank"]

        n = len(sorted_data)

        if n <= 25:
            critical_value = get_critical_value(n, alpha)

            if alternative == "two-sided":
                print("1. Hipotesis:")
                print(f"H0: Median sampel = {median_assumption}")
                print(f"H1: Median sampel ≠ {median_assumption}")

                print("2. Kriteria Uji:")
                print("Jika T < nilai kritis d, maka tolak H0")

                print(tabulate(table_data, headers=headers, tablefmt="grid"))

                print("\n3. Statistika Uji:")
                print(f"Jumlah peringkat positif: {T_positive}")
                print(f"Jumlah peringkat negatif: { T_negative}")

                T = min(T_positive, T_negative)

                print(f"Statistik uji Wilcoxon (T): {T}")
                print(f"Alpha: {alpha}")
                print(f"Nilai kritis Wilcoxon (d): {critical_value}")

                print("\n4. Penarikan Kesimpulan")

                if isinstance(critical_value, (int, float)) and T < critical_value:
                    print(f"Karena T: {T} < d: {critical_value}")
                    print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan antara median data dengan median yang disumsikan.")

                else:
                    print(f"Karena T: {T} >= d: {critical_value}")
                    print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan antara median data dengan median yang disumsikan.")

            elif alternative == "greater":
                print("1. Hipotesis:")
                print(f"H0: Median sampel ≤ {median_assumption}")
                print(f"H1: Median sampel > {median_assumption}")

                print(tabulate(table_data, headers=headers, tablefmt="grid"))

                print("2. Kriteria Uji:")
                print("Jika T- < nilai kritis d, maka tolak H0")

                print("\n3. Statistika Uji:")
                print(f"Jumlah peringkat positif: {T_positive}")
                print(f"Jumlah peringkat negatif: { T_negative}")

                print(f"Statistik uji Wilcoxon (T-): {T_negative}")
                print(f"Alpha: {alpha}")
                print(f"Nilai kritis Wilcoxon (d): {critical_value}")

                print("\n4. Penarikan Kesimpulan")

                if isinstance(critical_value, (int, float)) and T_negative < critical_value:
                    print(f"Karena T-: {T_negative} < d: {critical_value}")
                    print("Kesimpulan: Tolak H0, median yang diasumsikan lebih kecil daripada median data.")

                else:
                    print(f"Karena T: {T_negative} >= d: {critical_value}")
                    print("Kesimpulan: Gagal menolak H0, median yang diasumsikan lebih besar atau sama dengan median data.")

            elif alternative == "less":
                print("1. Hipotesis:")
                print(f"H0: Median sampel ≥ {median_assumption}")
                print(f"H1: Median sampel < {median_assumption}")

                print(tabulate(table_data, headers=headers, tablefmt="grid"))

                print("\n3. Statistika Uji:")
                print(f"Jumlah peringkat positif: {T_positive}")
                print(f"Jumlah peringkat negatif: { T_negative}")

                print(f"Statistik uji Wilcoxon (T+): {T_positive}")
                print(f"Alpha: {alpha}")
                print(f"Nilai kritis Wilcoxon (d): {critical_value}")

                print("\n4. Penarikan Kesimpulan")

                if isinstance(critical_value, (int, float)) and T_positive < critical_value:
                    print(f"Karena T+: {T_positive} < d: {critical_value}")
                    print("Kesimpulan: Tolak H0, median yang diasumsikan lebih besar daripada median data.")

                else:
                    print(f"Karena T: {T_positive} >= d: {critical_value}")
                    print("Kesimpulan: Gagal menolak H0, median yang diasumsikan lebih kecil atau sama dengan median data.")

            else:
                print("Pilihan alternative harus 'two-sided', 'greater', atau 'less'.")

        else:
            print("1. Hipotesis:")
            print("H0: Tidak ada perbedaan yang signifikan.")
            print("H1: Ada perbedaan yang signifikan.")

            print(tabulate(table_data, headers=headers, tablefmt="grid"))

            rank_counts = Counter(ranks)
            tied_ranks = [count for count in rank_counts.values() if count > 1]

            mu_T = n * (n + 1) / 4
            sigma_T = ((n * (n + 1) * (2 * n + 1)) / 24) ** 0.5

            if tied_ranks:
                sum_t3 = sum(t**3 for t in tied_ranks)  # Σ t³
                sum_t = sum(tied_ranks)  # Σ t
                correction = (sum_t3 - sum_t) / 48  # Koreksi

                std_T = ((n * (n + 1) * (2 * n + 1)) / 24 - correction) ** 0.5

            print("\n2. Kriteria Uji:")

            if alternative == "two-sided":
                print ("Jika |Z hitung| >= Z tabel, maka tolak H0")
                z_critical = norm.ppf(1 - alpha / 2)
                print(f"Z_tabel: ±{z_critical}")

                z = (min(T_negative, T_positive) - mu_T) / sigma_T

            elif alternative == "greater":
                print("Jika Z hitung > Z tabel, maka tolak H0")
                z_critical = norm.ppf(1 - alpha)
                print(f"Z_tabel: {z_critical}")

                z = (T_negative - mu_T) / sigma_T

            elif alternative == "less":
                print("Jika Z hitung < Z tabel, maka tolak H0")
                z_critical = norm.ppf(alpha)
                print(f"Z_tabel: {z_critical}")

                z = (T_positive - mu_T) / sigma_T

            else:
                print("Pilihan alternative harus 'two-sided', 'greater', atau 'less'.")

            print("\n3. Statistika Uji:")
            print(f"Jumlah peringkat positif: {T_positive}")
            print(f"Jumlah peringkat negatif: {T_negative}")
            # print(f"Statistik uji Wilcoxon (T): {T}")
            print(f"Alpha: {alpha}")

            print(f"Z_hitung: {z}, Z_tabel: ±{z_critical}")

            print("\n4. Penarikan Kesimpulan")

            if (alternative == "two-sided" and abs(z) >= z_critical) or \
               (alternative == "greater" and z > z_critical) or \
               (alternative == "less" and z < z_critical):
               print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan.")

            else:
                print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan.")


    elif test_type == "paired":
        if len(data1) != len(data2):
            raise ValueError("Panjang data1 dan data2 harus sama")

        print("1. Hipotesis:")
        print("H0: Tidak ada perbedaan yang signifikan antara kedua sampel")
        print("H1: Ada perbedaan yang signifikan antara kedua sampel")

        differences = [x - y for x, y in zip(data1, data2)]
        abs_differences = [abs(diff) for diff in differences]

        zero_data = [(d1, d2, di, abs_di) for d1, d2, di, abs_di in zip(data1, data2, differences, abs_differences) if di == 0]
        nonzero_data = [(d1, d2, di, abs_di) for d1, d2, di, abs_di in zip(data1, data2, differences, abs_differences) if di != 0]

        sorted_data = sorted(nonzero_data, key=lambda x: x[3])
        ranks = {}

        i = 1
        while i <= len(sorted_data):
            same_values = [sorted_data[i - 1][3]]
            j = i
            while j < len(sorted_data) and sorted_data[j][3] == sorted_data[i - 1][3]:
                same_values.append(sorted_data[j][3])
                j += 1
            avg_rank = sum(range(i, j + 1)) / len(same_values)
            for k in range(i, j + 1):
                ranks[k] = avg_rank
            i = j + 1

        table_data = []
        T_positive = 0
        T_negative = 0
        for i, (d1, d2, di, abs_di) in enumerate(sorted_data, start=1):
            rank = ranks[i]
            sign = '+' if di > 0 else '-'
            table_data.append([d1, d2, di, abs_di, rank, sign])
            if di > 0:
                T_positive += rank
            else:
                T_negative += rank

        for d1, d2, di, abs_di in zero_data:
            table_data.append([d1, d2, di, abs_di, '-', '-'])

        headers = ["Data1", "Data2", "di", "|di|", "Rank", "Signed Rank"]
        print(tabulate(table_data, headers=headers, tablefmt="grid"))

        T = min(T_positive, T_negative)
        n = len(nonzero_data)

        if n<= 25:
            print("2. Kriteria Uji:")
            print("Jika T < nilai kritis d, maka tolak H0")

            critical_value = get_critical_value(n, alpha)

            print("\n3. Statistika Uji:")
            print(f"Jumlah peringkat positif: {T_positive}")
            print(f"Jumlah peringkat negatif: {T_negative}")
            print(f"Statistik uji Wilcoxon (T): {T}")
            print(f"Alpha: {alpha}")
            print(f"Nilai kritis Wilcoxon (d): {critical_value}")

            print("\n4. Penarikan Kesimpulan")
            if isinstance(critical_value, (int, float)) and T < critical_value:
                print(f"Karena T: {T} < d: {critical_value}")
                print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan.")
            else:
                print(f"Karena T: {T} >= d: {critical_value}")
                print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan.")

        else:
            print("2. Kriteria Uji:")
            print("Jika |Z hitung| > Z tabel, maka tolak H0")

            rank_counts = Counter(ranks)
            tied_ranks = [count for count in rank_counts.values() if count > 1]

            mu_T = n * (n + 1) / 4
            sigma_T = ((n * (n + 1) * (2 * n + 1)) / 24) ** 0.5

            if tied_ranks:
                sum_t3 = sum(t**3 for t in tied_ranks)  # Σ t³
                sum_t = sum(tied_ranks)  # Σ t
                correction = (sum_t3 - sum_t) / 48  # Koreksi

                std_T = ((n * (n + 1) * (2 * n + 1)) / 24 - correction) ** 0.5


            z = (T - mu_T) / sigma_T
            z_critical = norm.ppf(1 - alpha / 2)

            print("\n3. Statistika Uji:")
            print(f"Jumlah peringkat positif: {T_positive}")
            print(f"Jumlah peringkat negatif: {T_negative}")
            print(f"Statistik uji Wilcoxon (T): {T}")
            print(f"Alpha: {alpha}")

            print(f"Z_hitung: {z}, Z_tabel: ±{z_critical}")

            print("\n4. Penarikan Kesimpulan")
            if abs(z) > z_critical:
                print(f"Karena absolute Z-hitung: {z} > Z-tabel {z_critical}")
                print("Kesimpulan: Tolak H0, ada perbedaan yang signifikan.")
            else:
                print(f"Karena absolute Z-hitung: {z} <= Z-tabel {z_critical}")
                print("Kesimpulan: Gagal menolak H0, tidak ada perbedaan yang signifikan.")

    else:
        print("Jenis uji tidak valid. Gunakan 'one_sample' atau 'paired'.")

### Test the Function

In [ ]:
test_type = 'one_sample'
data1 = [99, 100, 90, 94, 135, 108, 107, 111, 119, 104, 127, 109, 117, 105, 125]
median_assumption = 107
alpha = 0.05
wilcoxon_test(test_type, data1, median_assumption=median_assumption, alpha=alpha, alternative='two-sided')

1. Hipotesis:
H0: Median sampel = 107
H1: Median sampel ≠ 107
2. Kriteria Uji:
Jika T < nilai kritis d, maka tolak H0
+--------+------+--------+--------+---------------+
|   Data |   di |   |di| | Rank   | Signed Rank   |
+========+======+========+========+===============+
|    108 |    1 |      1 | 1.0    | +             |
+--------+------+--------+--------+---------------+
|    109 |    2 |      2 | 2.5    | +             |
+--------+------+--------+--------+---------------+
|    105 |   -2 |      2 | 2.5    | -             |
+--------+------+--------+--------+---------------+
|    104 |   -3 |      3 | 4.0    | -             |
+--------+------+--------+--------+---------------+
|    111 |    4 |      4 | 5.0    | +             |
+--------+------+--------+--------+---------------+
|    100 |   -7 |      7 | 6.0    | -             |
+--------+------+--------+--------+---------------+
|     99 |   -8 |      8 | 7.0    | -             |
+--------+------+--------+--------+---------------

In [ ]:
a = [83, 67, 80, 75, 86, 75, 81, 70, 68, 66,
     78, 82, 83, 74, 60, 61, 63, 79, 80, 81]
b = [71, 67, 72, 63, 83, 70, 69, 63, 59, 66,
     82, 82, 82, 80, 71, 73, 75, 70, 78, 69]
test_type = 'paired'
alpha = 0.05
wilcoxon_test(test_type, a, b, median_assumption, alpha)

1. Hipotesis:
H0: Tidak ada perbedaan yang signifikan antara kedua sampel
H1: Ada perbedaan yang signifikan antara kedua sampel
+---------+---------+------+--------+--------+---------------+
|   Data1 |   Data2 |   di |   |di| | Rank   | Signed Rank   |
+=========+=========+======+========+========+===============+
|      83 |      82 |    1 |      1 | 1.0    | +             |
+---------+---------+------+--------+--------+---------------+
|      80 |      78 |    2 |      2 | 2.0    | +             |
+---------+---------+------+--------+--------+---------------+
|      86 |      83 |    3 |      3 | 3.0    | +             |
+---------+---------+------+--------+--------+---------------+
|      78 |      82 |   -4 |      4 | 4.0    | -             |
+---------+---------+------+--------+--------+---------------+
|      75 |      70 |    5 |      5 | 5.0    | +             |
+---------+---------+------+--------+--------+---------------+
|      74 |      80 |   -6 |      6 | 6.0    | -     

## Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### PCOS Dataset

In [ ]:
!unzip /content/drive/MyDrive/dataset/pcos_dataset.zip -d /content/dataset/

Archive:  /content/drive/MyDrive/dataset/pcos_dataset.zip
  inflating: /content/dataset/pcos_dataset.csv  


In [ ]:
path = '/content/dataset/pcos_dataset.csv'
data1 = pd.read_csv(path)

In [ ]:
data1

,Age,BMI,Menstrual_Irregularity,Testosterone_Level(ng/dL),Antral_Follicle_Count,PCOS_Diagnosis
0,24,34.7,1,25.2,20,0
1,37,26.4,0,57.1,25,0
2,32,23.6,0,92.7,28,0
3,28,28.8,0,63.1,26,0
4,25,22.1,1,59.8,8,0
...,...,...,...,...,...,...
995,34,18.4,1,95.7,23,0
996,45,28.9,1,28.5,7,0
997,37,28.3,0,32.4,28,0
998,41,27.3,0,95.6,9,0


In [ ]:
data1.columns

Index(['Age', 'BMI', 'Menstrual_Irregularity', 'Testosterone_Level(ng/dL)',
       'Antral_Follicle_Count', 'PCOS_Diagnosis'],
      dtype='object')

In [ ]:
data1.dtypes

,0
Age,int64
BMI,float64
Menstrual_Irregularity,int64
Testosterone_Level(ng/dL),float64
Antral_Follicle_Count,int64
PCOS_Diagnosis,int64


In [ ]:
pcos_data = data1[data1['PCOS_Diagnosis'] == 1]
pcos_data

,Age,BMI,Menstrual_Irregularity,Testosterone_Level(ng/dL),Antral_Follicle_Count,PCOS_Diagnosis
13,38,33.2,1,43.0,21,1
14,21,26.1,1,83.4,13,1
27,18,26.6,1,42.2,21,1
28,29,29.7,1,98.7,14,1
33,34,32.9,1,72.3,24,1
...,...,...,...,...,...,...
975,44,28.4,1,68.3,16,1
978,37,33.3,1,96.1,20,1
983,32,32.0,1,71.9,18,1
988,34,25.7,1,74.1,28,1


In [ ]:
pcos_data['BMI'].mean()

30.126633165829144

In [ ]:
test_type = 'one_sample'
median_assumption = 30
alpha = 0.05
wilcoxon_test(test_type, pcos_data['BMI'], median_assumption=median_assumption, alpha=alpha, alternative = 'two-sided')

1. Hipotesis:
H0: Tidak ada perbedaan yang signifikan.
H1: Ada perbedaan yang signifikan.
+--------+------+--------+--------+---------------+
|   Data |   di |   |di| | Rank   | Signed Rank   |
+========+======+========+========+===============+
|   29.9 | -0.1 |    0.1 | 1.5    | -             |
+--------+------+--------+--------+---------------+
|   30.1 |  0.1 |    0.1 | 1.5    | +             |
+--------+------+--------+--------+---------------+
|   30.2 |  0.2 |    0.2 | 4.0    | +             |
+--------+------+--------+--------+---------------+
|   29.8 | -0.2 |    0.2 | 4.0    | -             |
+--------+------+--------+--------+---------------+
|   30.2 |  0.2 |    0.2 | 4.0    | +             |
+--------+------+--------+--------+---------------+
|   29.7 | -0.3 |    0.3 | 7.0    | -             |
+--------+------+--------+--------+---------------+
|   30.3 |  0.3 |    0.3 | 7.0    | +             |
+--------+------+--------+--------+---------------+
|   29.7 | -0.3 |    0.3 |

### Diabetes Health Indicator Dataset

In [ ]:
!unzip /content/drive/MyDrive/dataset/dhid_dataset.zip -d /content/dataset/

Archive:  /content/drive/MyDrive/dataset/dhid_dataset.zip
  inflating: /content/dataset/diabetes_012_health_indicators_BRFSS2015.csv  
  inflating: /content/dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv  
  inflating: /content/dataset/diabetes_binary_health_indicators_BRFSS2015.csv  


In [ ]:
path = '/content/dataset/diabetes_012_health_indicators_BRFSS2015.csv'
data = pd.read_csv(path)

In [ ]:
data

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,2.0,1.0,1.0,1.0,18.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [ ]:
data.columns

Index(['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [ ]:
data.dtypes

,0
Diabetes_012,float64
HighBP,float64
HighChol,float64
CholCheck,float64
BMI,float64
Smoker,float64
Stroke,float64
HeartDiseaseorAttack,float64
PhysActivity,float64
Fruits,float64


In [ ]:
diabetes_data = data[data['Diabetes_012'] == 2]
diabetes_data

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
8,2.0,1.0,1.0,1.0,30.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,5.0,30.0,30.0,1.0,0.0,9.0,5.0,1.0
10,2.0,0.0,0.0,1.0,25.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,13.0,6.0,8.0
13,2.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,4.0,6.0
17,2.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,2.0,0.0,0.0,0.0,1.0,7.0,5.0,6.0
23,2.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,13.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253659,2.0,0.0,1.0,1.0,37.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,0.0,0.0,6.0,4.0,1.0
253668,2.0,0.0,1.0,1.0,29.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,2.0,0.0,0.0,1.0,1.0,10.0,3.0,6.0
253670,2.0,1.0,1.0,1.0,25.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,5.0,15.0,0.0,1.0,0.0,13.0,6.0,4.0
253676,2.0,1.0,1.0,1.0,18.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0


## Apply Function

### PCOS Dataset

In [ ]:
test_type = 'one_sample'
median_assumption = 30
alpha = 0.05
wilcoxon_test(test_type, pcos_data['BMI'], median_assumption=median_assumption, alpha=alpha, alternative='two-sided')

1. Hipotesis:
H0: Tidak ada perbedaan yang signifikan.
H1: Ada perbedaan yang signifikan.
+--------+------+--------+--------+---------------+
|   Data |   di |   |di| | Rank   | Signed Rank   |
+========+======+========+========+===============+
|   29.9 | -0.1 |    0.1 | 1.5    | -             |
+--------+------+--------+--------+---------------+
|   30.1 |  0.1 |    0.1 | 1.5    | +             |
+--------+------+--------+--------+---------------+
|   30.2 |  0.2 |    0.2 | 4.0    | +             |
+--------+------+--------+--------+---------------+
|   29.8 | -0.2 |    0.2 | 4.0    | -             |
+--------+------+--------+--------+---------------+
|   30.2 |  0.2 |    0.2 | 4.0    | +             |
+--------+------+--------+--------+---------------+
|   29.7 | -0.3 |    0.3 | 7.0    | -             |
+--------+------+--------+--------+---------------+
|   30.3 |  0.3 |    0.3 | 7.0    | +             |
+--------+------+--------+--------+---------------+
|   29.7 | -0.3 |    0.3 |

### DHI Dataset

In [ ]:
test_type = 'one_sample'
median_assumption = 30
alpha = 0.05
wilcoxon_test(test_type, diabetes_data['BMI'], median_assumption=median_assumption, alpha=alpha, alternative='two-sided')

Output streaming akan dipotong hingga 5000 baris terakhir.
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +             |
+--------+------+--------+---------+---------------+
|     61 |   31 |     31 | 32860.5 | +  